In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.metrics import ndcg_score
from sklearn.model_selection import TimeSeriesSplit

# 1つ上のディレクトリにパスを通す
sys.path.append(os.path.abspath('../'))

from module.Dataset import read_preprocess_tsv, read_test_tsv, train_division, ColumnEncode

# 前処理済みデータの読み込み
train_A = read_preprocess_tsv('A')

# 予測用データの読み込み
test_A = read_test_tsv('A')

# エンコードクラスのインスタンス化
user_item_enc = ColumnEncode()
# エンコードの実行
train_A['user_id'] = user_item_enc.user_encode(train_A['user_id'])
train_A['product_id'] = user_item_enc.product_encode(train_A['product_id'])

test_A['user_id'] = user_item_enc.test_encode(test_A['user_id'])
# タイムスタンプを数値型に変換
train_A["time_stamp"] = pd.to_datetime(train_A["time_stamp"])
train_A['unix_time'] = train_A['time_stamp'].astype(np.int64) // 10**9  # UNIX時間に変換

# データ分割
train_X, train_y, valid_X, valid_y, features = train_division(train_A)

# 時系列交差検証
tscv = TimeSeriesSplit(n_splits=3)
model = xgb.XGBRegressor(objective='reg:squarederror')

for train_idx, val_idx in tscv.split(train_X):
    X_train, X_val = train_X.iloc[train_idx], train_X.iloc[val_idx]
    y_train, y_val = train_y.iloc[train_idx], train_y.iloc[val_idx]
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    print("Validation NDCG:", ndcg_score([y_val], [predictions]))

# 最終学習
model.fit(train_X, train_y)

# 推薦関数
def recommend_products(user_id, top_k=22):
    user_data = train_A[train_A['user_id'] == user_id][features]
    if user_data.empty:
        print(f"Warning: User {user_id} has no data.")
        return pd.DataFrame(columns=['product_id', 'rank'])
    predictions = model.predict(user_data)
    user_data['score'] = predictions
    recommendations = user_data.sort_values(by='score', ascending=False).head(top_k)
    recommendations['rank'] = range(1, len(recommendations) + 1)
    recommendations['user_id'] = user_item_enc.user_decode(recommendations['user_id'])
    recommendations['product_id'] = user_item_enc.product_decode(recommendations['product_id'])
    return recommendations[['user_id', 'product_id', 'rank']]

# nGCDの計算
def evaluate_ndcg():
    y_true = []
    y_score = []
    for user_id in test_A['user_id'].unique():
        if user_id in train_A['user_id'].values:
            actual = train_A[train_A['user_id'] == user_id].sort_values(by='relevance', ascending=False)['relevance'].values
            pred = recommend_products(user_id, top_k=len(actual))['rank'].values
            if len(pred) == 0:
                print(f"Warning: No predictions for user {user_id}.")
                continue
            y_true.append(actual)
            y_score.append(pred)
    return np.mean([ndcg_score([t], [s]) for t, s in zip(y_true, y_score)]) if y_true else 0.0

# 予測結果保存
def save_predictions():
    results = []
    for user_id in test_A['user_id'].unique():
        if user_id in train_A['user_id'].values:
            recommendations = recommend_products(user_id)
            for _, row in recommendations.iterrows():
                results.append([row['user_id'], row['product_id'], row['rank']])
    if not results:
        print("Warning: No predictions generated.")
    pd.DataFrame(results, columns=['user_id', 'product_id', 'rank']).to_csv('../dataset/predict_A.tsv', sep='\t', index=False)

# 推薦結果表示
print("nDCG Score:", evaluate_ndcg())

save_predictions()
